#  Прогнозирование заказов такси

После исправления методики прежние метрики требуют повторного расчёта. Сохранённые outputs очищены; историческая версия доступна в Git. Данные не включены в репозиторий. См. AUDIT.md.

Компания «Чётенькое такси» собрала исторические данные о заказах такси в аэропортах. Чтобы привлекать больше водителей в период пиковой нагрузки, нужно спрогнозировать количество заказов такси на следующий час. Постройте модель для такого предсказания.

Цель - построить модель для предсказания количества заказов такси на следующий час.

Значение метрики *RMSE* на тестовой выборке должно быть не больше 48.

Для достижения поставленной цели необходимо решить следующие задачи:

1. Загрузить данные и выполнить их ресемплирование по одному часу.
2. Проанализировать данные.
3. Обучить разные модели с различными гиперпараметрами. Сделать тестовую выборку размером 10% от исходных данных.
4. Проверить данные на тестовой выборке и сделать выводы.


<h1>Содержание<span class="tocSkip"></span></h1>
<li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка</a></span></li><li><span><a href="#Анализ" data-toc-modified-id="Анализ-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Анализ</a></span></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Обучение</a></span><ul class="toc-item"><li><span><a href="#Модель-линейной-регрессии" data-toc-modified-id="Модель-линейной-регрессии-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Модель линейной регрессии</a></span></li><li><span><a href="#Модель-решающего-дерева" data-toc-modified-id="Модель-решающего-дерева-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>Модель решающего дерева</a></span></li><li><span><a href="#Модель-XGBoost" data-toc-modified-id="Модель-XGBoost-3.3"><span class="toc-item-num">3.3&nbsp;&nbsp;</span>Модель XGBoost</a></span></li></ul></li><li><span><a href="#Тестирование" data-toc-modified-id="Тестирование-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Тестирование</a></span><ul class="toc-item"><li><span><a href="#Модель-линейной-регрессии" data-toc-modified-id="Модель-линейной-регрессии-4.1"><span class="toc-item-num">4.1&nbsp;&nbsp;</span>Модель линейной регрессии</a></span></li><li><span><a href="#Модель-решающего-дерева" data-toc-modified-id="Модель-решающего-дерева-4.2"><span class="toc-item-num">4.2&nbsp;&nbsp;</span>Модель решающего дерева</a></span></li><li><span><a href="#Модель-XGBoost" data-toc-modified-id="Модель-XGBoost-4.3"><span class="toc-item-num">4.3&nbsp;&nbsp;</span>Модель XGBoost</a></span></li><li><span><a href="#Сравнение-графиков" data-toc-modified-id="Сравнение-графиков-4.4"><span class="toc-item-num">4.4&nbsp;&nbsp;</span>Сравнение графиков</a></span></li>

## Подготовка

In [ ]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "research_checks.py").exists())
sys.path.insert(0, str(ROOT))
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data"))
# Импортируем все необходимые для работы библиотеки
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression 
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
from research_checks import taxi_features


In [ ]:
df = pd.read_csv(DATA_DIR / 'taxi.csv', index_col=0, parse_dates=[0])
if not df.index.is_unique:
    raise ValueError('Повторяющиеся временные метки требуют проверки источника')
df = df.sort_index()

In [ ]:
# Посмотрим на первые 10 строк нашей таблицы
df.head(10)

Видим, что в таблице имеется всего два столбца:

- datetime - время заказа (сразу через функцию index_col=[0] установили данный столбец в качестве индекса);
- num_orders - количество заказов.

In [ ]:
# Посмотрим на общую информацию о таблице
df.info()

Повторяющееся число заказов допустимо; дубликаты временных меток проверяются отдельно.

In [ ]:
df.index.is_monotonic_increasing

Данные расположены в хронологическом порядке.

In [ ]:
df = df.resample('1h').sum(min_count=1)
if df['num_orders'].isna().any():
    raise ValueError('Есть часы без наблюдений; нельзя автоматически считать их нулём')

На первом этапе загрузили таблицу с данными, сразу изменив формат столбца с датой и временем на нужный тип данных и установив его в качестве индекса. Данные не требовали никакой дополнительной обработки. Затем провели ресемплирование с шагом в 1 час, агрегировав значения по сумме.

## Анализ

На данном этапе проведем анализ изменения количества заявок во времени.

Построим график получения заявок за весь интервал времени в разрезе их суммарного количества по часам.

In [ ]:
df.plot(figsize=(10, 8))
plt.show()

По данному графику сложно сделать какой-то вывод. На мой взгляд, стоит отметить большие колебания количества заявок, особенно начиная приблизительно с середины августа. 

Чтобы лучше понимать временной ряд, разложим его на тренд и сезонную компоненту.

In [ ]:
# Функция для построения графиков
def plots(data):

    # График тренда
    decomposed = seasonal_decompose(data)
    plt.figure(figsize=(10, 8))
    plt.subplot(311)
    decomposed.trend.plot(ax=plt.gca())
    plt.title('Trend')
    # График сезонности
    plt.subplot(312)
    decomposed.seasonal.plot(ax=plt.gca())
    plt.title('Seasonality')
    # График остатков
    plt.subplot(313)
    decomposed.resid.plot(ax=plt.gca())
    plt.title('Residuals')
    plt.tight_layout()
    plt.show()

In [ ]:
# Применим функцию ко всему датафрейму
plots(df)

По графикам сезонности и остатков сложно сделать какие-либо выводы. По графику тренда видно, что наблюдается восходящий тренд почасовоого количества заявок на протяжении всего анализируемого периода.

Чтобы детальнее увидеть особенности распределения заявок по часам, выберем несколько дней датасета.

In [ ]:
df_new = df['2018-08-15':'2018-08-17']

In [ ]:
plots(df_new)

На данном интервале можно отследить наибольшую активность клиентов в течение дня. Наибольшее количество заявок наблюдается в вечернее время. Начиная с ночи количество начинает уменьшаться и лишь под утро заявок становтися больше.

В результате анализа удалось установить:

1. У компании наблюдается положительная тенденция роста количества заявок на протяжении всего анализиуемого периода.
2. Наибольшее количество заявок наблюдается в вечернее время, затем с ночи до утра происходит спад, после чего активность клиентов увеличивается.

## Обучение

На данном этапе проведем обучения моделей линейной регрессии, решающего дерева и XGBoost.

Перед началом обучения добавим новые признаки в датасет:

1. каледнарные признаки - year, month, day и dayofweek;
2. отстающие значения - lag;
3. скользящее среднее - rolling_mean.

Напишем функцию make_features, чтобы прибавить к таблице новые признаки.

In [ ]:
# Лаги и среднее используют только прошлые наблюдения; исходная таблица не меняется.
make_features = taxi_features

In [ ]:
new_df = make_features(df, 24, 100)

In [ ]:
new_df

Посмотрим на таблицу с новыми признаками.

In [ ]:
new_df.head(10)

Далее необходимо разбить выборку на тестовую и обучающую, а затем выделить фичи и таргет.

In [ ]:
train, test = train_test_split(new_df, shuffle=False, test_size=0.1)
train = train.dropna()

features_train = train.drop(['num_orders'], axis=1) 
target_train = train['num_orders']

features_test = test.drop(['num_orders'], axis=1) 
target_test = test['num_orders']

Так как мы имеем дело с временными рядами, то использовать обычную кросс-валидацию не получится в GridSearch. Поэтому применим TimeSeriesSplit.

In [ ]:
# Напишем функцию для вычисления RMSE модели
def RMSE(target, predict):
    return (mean_squared_error(target, predict))**0.5
   
# Создадим метрику RMSE с помощью функции make_scorer
RMSE_score = make_scorer(RMSE, greater_is_better=False)

### Модель линейной регрессии

In [ ]:
%%time

lnr = LinearRegression()
tss = TimeSeriesSplit(n_splits=3).split(features_train)
lnr.fit(features_train, target_train)
RMSE_lnr = cross_val_score(lnr, features_train, target_train, scoring=RMSE_score, 
                                      cv=tss).mean() * (-1)
print('Среднее значение RMSE модели линейной регрессии равно', RMSE_lnr)

### Модель решающего дерева

In [ ]:
%%time

dtr = DecisionTreeRegressor()
tss = TimeSeriesSplit(n_splits=3).split(features_train)
parametrs = {'criterion':['squared_error'],
            'max_depth': range(1, 13, 1),
            'random_state': [12345]}
grid_dtr = GridSearchCV(dtr, parametrs, scoring=RMSE_score, cv=tss)
grid_dtr.fit(features_train, target_train)

In [ ]:
# Выведем результат
grid_dtr.best_params_

Параметры определяются результатом GridSearchCV, а не вручную.

### Модель XGBoost

In [ ]:
%%time

xgb = XGBRegressor()
tss = TimeSeriesSplit(n_splits=3).split(features_train)
parametrs = {'max_depth': range(1, 10, 2),
            'learning_rate':[0.1, 0.3],
            'random_state': [12345],
            'n_estimators': range(10, 50, 10)}
grid_xgb = GridSearchCV(xgb, parametrs, scoring=RMSE_score, cv=tss)
grid_xgb.fit(features_train, target_train)

In [ ]:
# Выведем результат
grid_xgb.best_params_

Результат поиска гиперпараметров выводится выше.

Выбираем семейство модели по временному CV. Тест используется один раз после выбора.

## Однократная проверка на тесте

In [ ]:
cv_scores = {'linear': RMSE_lnr, 'tree': -grid_dtr.best_score_, 'xgb': -grid_xgb.best_score_}
selected = min(cv_scores, key=cv_scores.get)
models = {'linear': lnr, 'tree': grid_dtr.best_estimator_, 'xgb': grid_xgb.best_estimator_}
model = models[selected]
model.fit(features_train, target_train)
predictions = pd.Series(model.predict(features_test), index=target_test.index)
baseline = features_test['lag_24']
print('Модель:', selected, 'CV RMSE:', cv_scores[selected])
print('Test RMSE:', RMSE(target_test, predictions))
print('Сезонный baseline RMSE:', RMSE(target_test, baseline))
pd.DataFrame({'actual': target_test, 'prediction': predictions, 'lag24': baseline}).plot(figsize=(15, 5))
plt.show()


Протокол — прогноз на один час с известными фактическими значениями предыдущих часов (rolling origin). Это не прогноз сразу на весь тестовый горизонт. Порог RMSE ≤ 48 необходимо проверить повторным запуском. EDA всей истории имеет описательный характер; для нового эксперимента его следует ограничить обучающим периодом до выбора признаков.